# Messages, streaming, batching, and short-term memory

A model call can do more than return one complete answer. In this notebook, we send role-based messages, stream output as it arrives, batch independent requests, and keep a short conversation history.


In [9]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# Load GROQ_API_KEY from the local .env file.
load_dotenv()

groq_model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")


## Messages: text plus a label

A message is text plus a label saying who said it. The label lets the model distinguish application rules, a user question, and an earlier model reply.

Without labels, instructions and questions become one blob of text. With `SystemMessage` and `HumanMessage`, the model receives the intended structure. `AIMessage` lets us include an earlier reply when we continue a conversation.


In [10]:
# A HumanMessage carries both the text and the "human" label.
human_question = HumanMessage("How do I create a vegan chicken sandwich?")
print("Label:", human_question.type)
print("Text:", human_question.content)

# Without labels, the model receives one blob of instructions and question text.
plain_text_response = groq_model.invoke(
    "You are a street chef. How do I create a vegan chicken sandwich?"
)
print(plain_text_response.text)

# With labels, rules and the user's question are separate messages.
message_response = groq_model.invoke(
    [
        SystemMessage("You are a street chef."),  # application rules
        HumanMessage("How do I create a vegan chicken sandwich?"),  # user question
    ],
    max_tokens=200,  # limit the response length
)
print(message_response.text)


Label: human
Text: How do I create a vegan chicken sandwich?
**Vegan “Chicken” Sandwich – Street‑Food Style**  
*(Makes 2 hearty sandwiches – perfect for a food‑truck window or a pop‑up stand)*  

---

## 🌟 What makes it “chicken‑like”?

- **Texture:** We’ll use seasoned soy‑based “chicken” strips (or a homemade chickpea‑flax “fillet”) that stay crispy on the outside and juicy inside.  
- **Flavor:** A quick brine + a smoky‑spicy coating gives that familiar umami bite.  
- **Street‑ready:** Everything can be pre‑made, assembled fast, and wrapped in parchment or a paper bag for that on‑the‑go vibe.

---

## 🛒 Ingredients

| Category | Item | Qty (for 2 sandwiches) |
|----------|------|------------------------|
| **Vegan “Chicken”** | Soy‑based chicken strips (store‑bought, e.g., *Beyond Chicken* or *Gardein*), or homemade chickpea “fillet” (see note) | 200 g (≈ 7 oz) |
| **Marinade** | Light soy sauce | 2 Tbsp |
| | Apple cider vinegar | 1 Tbsp |
| | Maple syrup or agave | 1 Tbsp |
| | 

In [12]:
print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
# An AIMessage can carry an earlier reply into the next model call.
conversation_with_history = [
    SystemMessage("You are a street chef."),
    HumanMessage("How do I create a vegan chicken sandwich?"),
    AIMessage(
        "First, you'll need to find a vegan chicken. And fry it "
    ),  # earlier model reply
    # note we are assuming this is the model's response to the previous question.
    HumanMessage("can I create a bread from it?"),  
]

history_response = groq_model.invoke(conversation_with_history)
print(history_response.text)

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
### Making a Savory “Vegan Chicken” Bread (Stuffed Rolls)

You can absolutely turn your vegan‑chicken filling into a delicious, handheld bread that’s perfect for a street‑food stall or a quick snack at home. Below is a straightforward recipe for **soft, fluffy rolls stuffed with seasoned vegan chicken**. The dough is a classic yeasted bread, and the filling is the same “vegan chicken” you’d use for a sandwich.

---

## 1. Gather Your Ingredients  

### For the Dough  
| Ingredient | Amount | Notes |
|------------|--------|-------|
| Bread flour (or all‑purpose) | 4 cups (500 g) | Gives structure & chew |
| Warm water (110 °F / 43 °C) | 1 ½ cups (360 ml) | Adjust if dough feels too dry |
| Active dry yeast | 2 ½ tsp (1 packet) | |
| Sugar | 2 tbsp | Feeds the yeast |
| Salt | 1 ½ tsp | |
| Olive oil (or neutral oil) | 2 tbsp | Adds tenderness |
| Optional: 1 tbsp milk‑free milk (soy, oat) | | Makes crust a bit softer |

### For the Vegan‑Chicken 

## Stream a response

Streaming lets the application show text as the model generates it. This makes long answers feel more responsive, even though the final answer is still produced token by token.


In [ ]:
stream_prompt = "Explain why API keys should not be committed to Git in 10 sentences."

for chunk in groq_model.stream(stream_prompt):
    # Each chunk contains the next available piece of the model response.
    print(chunk.text, end="", flush=True)
print()


# You will see the model's response printed in real-time as it is generated, rather than waiting for the entire response to be completed. This is useful for long responses or when you want to provide immediate feedback to users.

1. API keys are secret credentials that grant access to services, and exposing them publicly allows anyone to use those services on your behalf.  
2. When a key is committed to a Git repository, especially a public one, it becomes part of the repository’s history and is difficult to remove completely.  
3. Attackers can scan public Git histories for patterns that look like keys, automatically harvesting them for malicious use.  
4. Unauthorized usage can quickly exhaust rate limits, incur unexpected costs, or cause service outages for legitimate users.  
5. Leaked keys can also expose sensitive data stored behind the API, violating privacy regulations and damaging user trust.  
6. Many platforms automatically revoke or rotate keys once a leak is detected, which forces developers to spend time re‑issuing and updating credentials.  
7. Committing keys undermines the principle of least privilege, as the same key may be used across multiple environments (development, staging, production). 

## Batch independent requests

Batching is useful when prompts do not depend on each other. It lets the integration schedule multiple model calls together; it does not create a conversation between those prompts.


In [13]:
batch_prompts = [
    "Give one use for a low temperature setting.",
    "Give one use for a high temperature setting.",
    "Define an LLM in one sentence.",
]

batch_responses = groq_model.batch(batch_prompts)
for prompt, batch_response in zip(batch_prompts, batch_responses):
    print(f"Prompt: {prompt}")
    print(f"Answer: {batch_response.text}")
    print()


Prompt: Give one use for a low temperature setting.
Answer: A low temperature setting is ideal for **refrigerating perishable foods**—it keeps items like fruits, vegetables, dairy, and leftovers chilled (typically around 35–40 °F or 2–4 °C) to slow bacterial growth and extend their freshness.

Prompt: Give one use for a high temperature setting.
Answer: One common use for a high temperature setting is **baking bread**—the oven needs to reach around 425 °F (220 °C) to create the burst of steam that gives the crust its crispness and helps the dough rise quickly.

Prompt: Define an LLM in one sentence.
Answer: A large language model (LLM) is a deep‑learning neural network trained on massive text corpora that generates or understands human language by predicting the next word (or token) in a sequence.



## Keep short-term memory yourself

A chat model remembers only the messages included in the current request. This example stores the conversation in a Python list and sends it again on the next turn. Later lessons will use more durable memory patterns.


In [ ]:
conversation = [
    SystemMessage("You are a concise assistant for a Python course."),
    HumanMessage("My name is Ada. I am learning about LLMs."),
]

first_response = groq_model.invoke(conversation)
conversation.append(first_response)
conversation.append(HumanMessage("What is my name and what am I learning?"))

second_response = groq_model.invoke(conversation)
print(second_response.text)
